# Module 4 CARE Principles and Data Sovereignty Framework

Modules 1–3 ran under a governance context that permitted PUBLIC tier only. This
module is where that context is built, examined, and if the relationship
supports it extended.

**The claim this repo makes:** governance is not a compliance appendix. It is a
pipeline stage that raises exceptions, a default that withholds, and an audit log
that records what was attempted. Prose commitments get skipped under deadline;
a `PermissionError` does not.

The honest limit of that claim, stated up front: none of this enforces anything
against an analyst who deletes the check. It is not security. It is a design that
makes the governance decision **visible at the point of use** you have to write
the line that says "yes, this is approved, and here is the record" instead of
letting silence default to release.

Five parts:

1. CARE, and what FAIR omits
2. Tiered publication, and why community tier is the finest resolution
3. Data-use agreements as structured objects
4. Provenance, including knowledge attribution
5. The publication gate, which blocks by default

In [1]:
import pandas as pd

from daear_toolkit import sovereignty as sv

NATION = "Oglala Sioux Tribe"
ctx = sv.GovernanceContext(nation=NATION)

print("CARE and FAIR are not alternatives. CARE is what FAIR omits.\n")
print("FAIR asks whether data can be found and reused. It says nothing about whether")
print("it SHOULD be, by whom, or who benefits. An open dataset about a community that")
print("the community did not consent to and cannot control is maximally FAIR and a")
print("complete CARE failure.\n")
print(sv.ocap_note())

CARE and FAIR are not alternatives. CARE is what FAIR omits.

FAIR asks whether data can be found and reused. It says nothing about whether
it SHOULD be, by whom, or who benefits. An open dataset about a community that
the community did not consent to and cannot control is maximally FAIR and a
complete CARE failure.

OCAP(R) -- Ownership, Control, Access, and Possession -- is a registered trademark of the First Nations Information Governance Centre (FNIGC). It was developed in and for a First Nations context in Canada. It is referenced here because its principles are widely applied, and it should not be assumed to be adopted by, or appropriate for, any given Tribal Nation in the United States without that Nation saying so. Several Nations have their own research codes and IRBs that take precedence.


## CARE assessment

Unanswered questions count as **unmet**, not as neutral. An unexamined
governance question is an unmet one, and scoring it neutral produces a
comfortable middling number that means nothing.

Run this honestly. A low score early in a relationship is the expected and
correct result.

In [2]:
# Answers reflect the actual state of this demo repo: public data, no agreement,
# no community relationship established for this specific analysis.
responses = {
    "Does the Nation define what benefit means here, or does the analyst?": False,
    "Does a usable product reach the community, or only a publication and a repo?": False,
    "Is capacity transferred does someone in the Nation end up able to run and modify this?": None,
    "Who holds the outputs when the funding ends?": None,

    "Has the Nation's designated authority reviewed the questions being asked, not just the methods?": False,
    "Can the Nation halt, amend, or withdraw from the work after it has started?": None,
    "Who decides what is published, at what resolution, and in what venue?": True,
    "Are derived products and models covered by the agreement, or only the raw data?": None,

    "Are limitations stated in terms a non-specialist reader can act on?": True,
    "Is the analysis reviewed by people with ground knowledge before release?": False,
    "Have foreseeable harms from misinterpretation been named, including by third parties?": True,
    "Is there a correction and retraction path if the analysis turns out to be wrong?": None,

    "Does the framing avoid deficit narratives -- is the community described as capable?": True,
    "Are Indigenous knowledge contributions attributed as knowledge, or extracted as 'input data'?": True,
    "Would the community recognize itself in how it is described?": None,
    "Is the relationship ongoing, or does it end at the deliverable?": None,
}

report = sv.care_assessment(responses)

CARE PRINCIPLES ASSESSMENT

Collective Benefit: 0/4
   NOT MET      Does the Nation define what benefit means here, or does the analyst?
   NOT MET      Does a usable product reach the community, or only a publication and a repo?
   UNADDRESSED  Is capacity transferred -- does someone in the Nation end up able to run and modify this?
   UNADDRESSED  Who holds the outputs when the funding ends?

Authority to Control: 1/4
   NOT MET      Has the Nation's designated authority reviewed the questions being asked, not just the methods?
   UNADDRESSED  Can the Nation halt, amend, or withdraw from the work after it has started?
   UNADDRESSED  Are derived products and models covered by the agreement, or only the raw data?

Responsibility: 2/4
   NOT MET      Is the analysis reviewed by people with ground knowledge before release?
   UNADDRESSED  Is there a correction and retraction path if the analysis turns out to be wrong?

Ethics: 2/4
   UNADDRESSED  Would the community recognize itself in 

### Reading the result

A low score here is the correct outcome for a demo repo with no community
relationship behind it, and reporting it honestly is more useful than
manufacturing a high one.

The pattern in the failures is the informative part. This repo does reasonably on
**Responsibility** and **Ethics** the things an analyst can do alone: state
limitations, avoid deficit framing, attribute knowledge properly. It fails on
**Collective Benefit** and **Authority to Control** the things that require a
relationship.

That asymmetry is not a gap to be closed with better code. It is the accurate
shape of what analysis can and cannot supply on its own.

## Publication tiers

Note the ordering: **higher tier = finer resolution AND tighter control.**
Communities receive more detail about their own lands, not less.

The inverted version: public tier detailed, community receiving a summary is
a surprisingly common accident. It happens whenever the public product is built
first and the community version is derived from it by subtraction.

In [3]:
rows = []
for tier in sv.Tier:
    r = sv.TIER_RULES[tier]
    rows.append({"tier": tier.name, "level": int(tier), **r})
tiers = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 60)
tiers

,tier,level,spatial,temporal,audience,requires
0,PUBLIC,0,jurisdiction-level aggregate (whole reservation or larger),decadal or multi-year,"open repository, publication, accelerator demo",public federal data sources only; Nation notified before...
1,PARTNER,1,"sub-jurisdictional (district, watershed, community area)",annual,named agency or academic partners,executed data-use agreement naming the partner and the p...
2,COMMUNITY,2,native raster resolution (10-30 m),daily to seasonal,the Nation whose lands are described,Nation holds the data and the release authority
3,RESTRICTED,3,not applicable -- not stored,not applicable,knowledge holders as determined by the Nation,these data do not enter analytical systems; see Cultural...


In [4]:
print("Gating behaviour under the current context (no agreement):\n")
for tier in sv.Tier:
    try:
        ctx.check(tier, dataset="mtbs", action="demonstrate")
        print(f"  {tier.name:<11} PERMITTED")
    except PermissionError as e:
        print(f"  {tier.name:<11} DENIED -- {str(e).splitlines()[0][:70]}")

print("\nRESTRICTED denies even with a maximal agreement. That tier is not a permission")
print("level, it marks data that does not enter analytical systems at all, because the")
print("protection is in not holding it. No agreement can grant what the design refuses.")

Gating behaviour under the current context (no agreement):

  PUBLIC      PERMITTED
  PARTNER     DENIED -- PARTNER-tier access requires an executed data-use agreement with Oglal
  COMMUNITY   DENIED -- COMMUNITY-tier access requires an executed data-use agreement with Ogl
  RESTRICTED  DENIED -- RESTRICTED-tier data (cultural resources, sacred sites, burial grounds

RESTRICTED denies even with a maximal agreement. That tier is not a permission
level, it marks data that does not enter analytical systems at all, because the
protection is in not holding it. No agreement can grant what the design refuses.


## Agreements as structured objects

Every field is required except notes and restrictions. An agreement that cannot
name its parties, purpose, expiry, and release authority is not an agreement,
it is an understanding, and understandings are what people remember differently
two years later when someone wants to publish.

**Expiry is required.** Open-ended consent is not consent; it is a permission
someone gave once, to a project that has since changed, that nobody can revisit
because there is no renewal moment.

In [5]:
# Illustrative structure. This agreement does not exist.
example = sv.DataUseAgreement(
    nation=NATION,
    counterparty="Daear Consulting LLC",
    purpose="Post-fire watershed and fuels assessment supporting Tribal fire program planning",
    executed="2026-09-01",
    expires="2028-09-01",
    release_authority="OST Natural Resources Regulatory Agency, per Tribal Council resolution",
    approved_tiers=(sv.Tier.PUBLIC, sv.Tier.PARTNER, sv.Tier.COMMUNITY),
    data_scope=("mtbs", "fod", "census_aiannh", "bia_lar", "sentinel2", "landfire"),
    restrictions=(
        "No publication of sub-district resolution outputs without separate written approval",
        "No cultural resource data of any kind enters the analysis",
        "Derived models and training data are covered by this agreement, not only raw inputs",
        "Nation may halt the work and require deletion of derived products at any time",
    ),
    notes="ILLUSTRATIVE STRUCTURE ONLY: this agreement has not been executed.",
)

print(example.summary())
print(f"\nDays until expiry: {example.days_until_expiry:+d}")
print("\nRestrictions carried with the agreement:")
for r in example.restrictions:
    print(f"  - {r}")

print("\nNote the third restriction. Agreements that cover 'the data' but are silent on")
print("derived models are a live gap: a model trained on community data and then")
print("deployed elsewhere has taken the data with it in a form the agreement never named.")

Oglala Sioux Tribe <-> Daear Consulting LLC | CURRENT (expires 2028-09-01, +761 days) | tiers: ['PUBLIC', 'PARTNER', 'COMMUNITY']

Days until expiry: +761

Restrictions carried with the agreement:
  - No publication of sub-district resolution outputs without separate written approval
  - No cultural resource data of any kind enters the analysis
  - Derived models and training data are covered by this agreement, not only raw inputs
  - Nation may halt the work and require deletion of derived products at any time

Note the third restriction. Agreements that cover 'the data' but are silent on
derived models are a live gap: a model trained on community data and then
deployed elsewhere has taken the data with it in a form the agreement never named.


In [6]:
# Scope is enumerated, not open-ended.
ctx_agreed = sv.GovernanceContext(nation=NATION, agreement=example)

for dataset in ("mtbs", "housing_authority_records"):
    try:
        ctx_agreed.check(sv.Tier.COMMUNITY, dataset=dataset, action="demonstrate scope")
        print(f"COMMUNITY tier, '{dataset}': PERMITTED")
    except PermissionError as e:
        print(f"COMMUNITY tier, '{dataset}': DENIED {str(e).splitlines()[0][:80]}")

print("\nAn agreement covering fire history does not silently extend to housing records")
print("because both are 'fire related'. Scope creep is the most common way a good-faith")
print("agreement stops describing what is actually happening.")

COMMUNITY tier, 'mtbs': PERMITTED
COMMUNITY tier, 'housing_authority_records': DENIED Not permitted: dataset 'housing_authority_records' is outside the agreed scope. 

An agreement covering fire history does not silently extend to housing records
because both are 'fire related'. Scope creep is the most common way a good-faith
agreement stops describing what is actually happening.


## Data provenance and knowledge attribution

The field that distinguishes this from ordinary data lineage is
`indigenous_knowledge_contributions`.

Standard provenance tracks datasets and code. When Indigenous knowledge shapes an
analysis including which watersheds matter, what counts as a healthy landscape, where
fire historically moved and that contribution is typically absorbed into "the
method" and disappears from the record. Naming it is both an attribution
obligation and a factual correction to the lineage.

In [7]:
prov = sv.ProvenanceRecord(
    product="Pine Ridge fire history and exposure summary",
    created="2026-07-26",
    creator="Daear Consulting LLC",
    nation=NATION,
    data_sources=(
        "MTBS burn severity and perimeters, 1984-2024 (USGS/USFS)",
        "Short, K.C. 2022, FPA_FOD_20221014, RDS-2013-0009.6 (USDA FS Research Data Archive)",
        "U.S. Census Bureau TIGERweb AIANNH boundaries",
        "BIA Land Area Representations",
        "OpenStreetMap building footprints (ODbL)",
    ),
    methods=(
        "Fire rotation from mapped perimeter area",
        "Ignition cause profile with undetermined fraction reported",
        "Jurisdictional complexity on a 5 km grid",
        "Asset-based adaptive capacity framework (components unpopulated)",
    ),
    tier=sv.Tier.PUBLIC,
    agreement_ref=None,
    indigenous_knowledge_contributions=(
        "PLACEHOLDER: no knowledge contributions have been incorporated in this demo. "
        "Where they are, name the contributor and the nature of the contribution, with "
        "their consent to be named.",
    ),
    limitations=(
        "Satellite-era record (1984-) is a suppression-era baseline, not a natural one",
        "OSM structure counts are lower bounds; undercount likely worse on reservation lands",
        "Cultural resources deliberately absent so any values-at-risk total undercounts",
        "Adaptive capacity component values are fabricated structural examples",
    ),
)

print(prov.to_markdown())
with open("../outputs/04_provenance.md", "w") as fh:
    fh.write(prov.to_markdown())

# Provenance: Pine Ridge fire history and exposure summary

- **Created:** 2026-07-26 by Daear Consulting LLC
- **Concerning:** Oglala Sioux Tribe
- **Publication tier:** PUBLIC
- **Agreement:** none (public federal data, PUBLIC tier)
- **Fingerprint:** `cbc71507d6b2f4c6`

## Data sources
- MTBS burn severity and perimeters, 1984-2024 (USGS/USFS)
- Short, K.C. 2022, FPA_FOD_20221014, RDS-2013-0009.6 (USDA FS Research Data Archive)
- U.S. Census Bureau TIGERweb AIANNH boundaries
- BIA Land Area Representations
- OpenStreetMap building footprints (ODbL)

## Methods
- Fire rotation from mapped perimeter area
- Ignition cause profile with undetermined fraction reported
- Jurisdictional complexity on a 5 km grid
- Asset-based adaptive capacity framework (components unpopulated)

## Indigenous knowledge contributions
*Attributed as knowledge contributions, not as input data.*

- PLACEHOLDER: no knowledge contributions have been incorporated in this demo. Where they are, name the contributor 

## The publication gate

The check that blocks by default.

Federal open data as an input does not make the **output** public. An analysis
about a Nation's lands is a new artifact and its release is a governance
decision that source licensing does not settle.

The gate requires two things: approval, and a record of how it was obtained. An
unattributed approval cannot be verified later and will not survive a partner
asking who signed off.

In [8]:
try:
    ctx.check_publication("Pine Ridge fire history and exposure analysis")
except PermissionError as e:
    print("BLOCKED:\n")
    print(e)

BLOCKED:

Public release of Pine Ridge fire history and exposure analysis concerning Oglala Sioux Tribe is not approved in this context.

Federal open data as an input does not make the OUTPUT public by default. An analysis about a Nation's lands is a new artifact and its release is a governance decision.

To proceed, obtain sign-off from the Nation's designated authority and record it:
    ctx.public_release_approved = True
    ctx.approval_record = '<who approved, when, in what forum>'

If you are about to set these without having obtained sign-off, that is the moment this check exists for.


In [9]:
# Approval without a record is also blocked setting the flag is not enough.
ctx.public_release_approved = True
try:
    ctx.check_publication()
except PermissionError as e:
    print("STILL BLOCKED:\n")
    print(e)

# Reset: this demo has no approval and should not pretend otherwise.
ctx.public_release_approved = False
print("\n\nContext reset to unapproved this repo has no sign-off recorded.")

STILL BLOCKED:

public_release_approved is set but approval_record is empty. Record who approved release, when, and in what forum -- an unattributed approval cannot be verified later and will not survive a partner asking.


Context reset to unapproved this repo has no sign-off recorded.


In [10]:
print("Audit log for this session: every gated operation, permitted or denied:\n")
log = ctx.audit_log()
print(log.to_string(index=False))
log.to_csv("../outputs/04_governance_audit_log.csv", index=False)

Audit log for this session: every gated operation, permitted or denied:

               when       tier dataset      action    result
2026-08-02T11:36:45     PUBLIC    mtbs demonstrate PERMITTED
2026-08-02T11:36:45    PARTNER    mtbs demonstrate    DENIED
2026-08-02T11:36:45  COMMUNITY    mtbs demonstrate    DENIED
2026-08-02T11:36:45 RESTRICTED    mtbs demonstrate    DENIED


## Before this repository goes public

A checklist, not a formality. Items 1–3 gate release.

1. **Notify the Nation.** Even for public federal data, an analysis published
   about Pine Ridge should not first reach the Oglala Sioux Tribe as a link
   someone else sent them.
2. **Obtain and record sign-off** from the designated authority OST Natural
   Resources Regulatory Agency or as the Nation directs, then set
   `ctx.public_release_approved` and `ctx.approval_record`.
3. **Have it read by someone with ground knowledge.** The fire history framing in
   Module 1 makes claims about cultural burning and suppression that should be
   checked by people who hold that history, not inferred from literature.
4. **Consider whether Pine Ridge should be the demo region at all.** A
   non-Tribal analog landscape would demonstrate identical capability with none
   of this weight. Naming a specific Nation buys realism; it costs a governance
   obligation that a demo may not be able to honour. That trade is a real
   decision and it should be made deliberately rather than by default.
5. **Confirm no derived product carries finer resolution than PUBLIC tier**, check
   the saved outputs, not just the notebook.

## Summary: what this framework does and does not do

**Does:**
- Fails closed. The default is withhold, and permission must be written.
- Makes governance visible at the point of use rather than in a document nobody
  opens during a deadline week.
- Records what was attempted, so the audit log exists whether or not anyone
  planned to keep one.
- Refuses some things categorically cultural resource data does not enter the
  pipeline at any tier, and no agreement can grant what the design refuses.

**Does not:**
- Enforce anything against someone who deletes the check. It is not security.
- Substitute for a relationship. Every function here assumes one exists and gives
  it structure; none creates one.
- Tell you what a Nation would consent to. No code can, and code that appeared
  to would be worse than none.

**Why it belongs in the pitch.** Most competitors treat data governance as a
compliance section written after the technical work. Shipping it as executable
code that blocks by default and being willing to show a CARE assessment this
repo scores poorly on is a more credible position than a polished claim of
compliance. It is also the part that cannot be replicated by a team without the
relationships, which is the actual moat.